# 01 자료·환경 점검 (후보 0)
후보 0(오믹스 분석)에 쓸 기존 자료가 있는지, venv 패키지와 R이 준비됐는지 확인한다. 이미지는 쓰지 않는다.
- 커널: `C:\Users\Admin\EXAONEPath\exaonepath-venv`
- 출력: `C:\Users\Admin\EXAONEPath_data\v5\01_inventory\`

In [7]:
# 1. 경로
from pathlib import Path
import sys, shutil, subprocess, json
from importlib.metadata import version, PackageNotFoundError
import pandas as pd

V3  = Path(r"C:\Users\Admin\EXAONEPath_data\v3")
V4  = Path(r"C:\Users\Admin\EXAONEPath_data\v4")
SH  = Path(r"C:\Users\Admin\EXAONEPath_data\_shared")
OUT = Path(r"C:\Users\Admin\EXAONEPath_data\v5\01_inventory")
OUT.mkdir(parents=True, exist_ok=True)

In [8]:
# 2. 기존 자료 존재 확인
key  = [SH / f for f in ("tcga_cdr.xlsx", "absolute_purity.tsv", "pancan_clinical.tsv", "sample_quality.tsv",
                         "thorsson_mmc2.xlsx", "estimate_genesets.gmt", "HM450.hg38.manifest.tsv.gz",
                         r"epidish_refs\centEpiFibIC.m.rda")]
key += [V3 / p.format(c=c) for c in ("luad", "lusc") for p in
        (r"omics\{c}_rna_tpm.parquet", r"omics\{c}_meth450_beta.h5", r"labels\{c}_patient_table.csv")]
key += [V4 / p for p in (r"03_gdc_inventory\gdc_files.csv", r"04_pancan_tables\files_used.csv",
                         r"04_pancan_tables\raw\MAF", r"04_pancan_tables\raw\CNV", r"04_pancan_tables\raw\RPPA")]

size = lambda p: sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) if p.is_dir() else p.stat().st_size
kf = pd.DataFrame({"path": [str(p) for p in key], "exists": [p.exists() for p in key],
                   "MB": [round(size(p) / 2**20, 1) if p.exists() else None for p in key],
                   "files": [sum(1 for f in p.rglob("*") if f.is_file()) if p.is_dir() else None for p in key]})
kf.to_csv(OUT / "key_files.csv", index=False, encoding="utf-8-sig")
print(f"존재 {kf.exists.sum()}/{len(kf)}")
kf

존재 19/19


,path,exists,MB,files
0,C:\Users\Admin\EXAONEPath_data\_shared\tcga_cd...,True,2.8,NaN
1,C:\Users\Admin\EXAONEPath_data\_shared\absolut...,True,0.9,NaN
2,C:\Users\Admin\EXAONEPath_data\_shared\pancan_...,True,17.8,NaN
3,C:\Users\Admin\EXAONEPath_data\_shared\sample_...,True,8.1,NaN
4,C:\Users\Admin\EXAONEPath_data\_shared\thorsso...,True,6.2,NaN
5,C:\Users\Admin\EXAONEPath_data\_shared\estimat...,True,0.0,NaN
6,C:\Users\Admin\EXAONEPath_data\_shared\HM450.h...,True,21.6,NaN
7,C:\Users\Admin\EXAONEPath_data\_shared\epidish...,True,0.0,NaN
8,C:\Users\Admin\EXAONEPath_data\v3\omics\luad_r...,True,85.8,NaN
9,C:\Users\Admin\EXAONEPath_data\v3\omics\luad_m...,True,263.4,NaN


In [9]:
# 3. venv 패키지, R, 디스크
PKGS = ["numpy", "pandas", "scipy", "statsmodels", "lifelines", "scikit-learn", "gseapy", "pyarrow", "h5py",
        "openpyxl", "pyreadr", "requests", "matplotlib", "seaborn", "upsetplot"]
def ver(p):
    try: return version(p)
    except PackageNotFoundError: return None
pk = pd.DataFrame({"package": PKGS, "version": [ver(p) for p in PKGS]})
missing = pk[pk.version.isna()].package.tolist()

rscript = shutil.which("Rscript")
r_ver = subprocess.run([rscript, "--version"], capture_output=True, text=True).stderr.strip() if rscript else None
free_gb = round(shutil.disk_usage("C:\\").free / 2**30, 1)

print("Python", sys.version.split()[0], "|", sys.executable)
print("미설치:", missing or "없음")
print("Rscript:", rscript or "없음", "|", r_ver)
print("C: 여유", free_gb, "GB")
if missing:
    print("설치 명령:", f'"{sys.executable}" -m pip install ' + " ".join(missing))
(OUT / "env_freeze.txt").write_text(
    subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout, encoding="utf-8")
pk

Python 3.14.6 | C:\Users\Admin\EXAONEPath\exaonepath-venv\Scripts\python.exe
미설치: ['upsetplot']
Rscript: 없음 | None
C: 여유 116.2 GB
설치 명령: "C:\Users\Admin\EXAONEPath\exaonepath-venv\Scripts\python.exe" -m pip install upsetplot


,package,version
0,numpy,2.5.2
1,pandas,3.0.5
2,scipy,1.18.1
3,statsmodels,0.15.0
4,lifelines,0.30.0
5,scikit-learn,1.9.1
6,gseapy,1.3.1
7,pyarrow,26.0.0
8,h5py,3.16.0
9,openpyxl,3.1.5


In [10]:
# 4. 요약 저장
summary = {"missing_inputs": kf.loc[~kf.exists, "path"].tolist(), "missing_packages": missing,
           "rscript": rscript, "disk_free_gb": free_gb}
(OUT / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
summary

{'missing_inputs': [],
 'missing_packages': ['upsetplot'],
 'rscript': None,
 'disk_free_gb': 116.2}